# Reddit polling and Spark Structured Streaming

Original coursework with security and small integration fixes. Outputs cleared. Requires your own Reddit API access; running makes live API requests. See README.md and the root EDITS.md.


In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, from_unixtime
import praw
import csv
import os
import time
import threading
from datetime import datetime

# Initialize SparkSession
spark = SparkSession.builder \
    .appName("LiveRedditData") \
    .master("local[2]") \
    .config("spark.driver.bindAddress", "127.0.0.1") \
    .config("spark.driver.port", "4040") \
    .getOrCreate()

client_id = os.environ["REDDIT_CLIENT_ID"]
client_secret = os.environ["REDDIT_CLIENT_SECRET"]
user_agent = os.environ["REDDIT_USER_AGENT"]

# Initialize PRAW
reddit = praw.Reddit(
    client_id=client_id,
    client_secret=client_secret,
    user_agent=user_agent
)

# Directory for streaming data
output_dir = "reddit_stream_output"
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

# Set parameters for streaming
my_subreddits = ["news","worldnews","climate","geography","weather"]

keywords = ["extreme weather", "severe weather", "weather emergency", 
 "natural disaster", "hurricane", "cyclone", 
 "typhoon", "flood", "wildfire", "disaster relief"]

processed_post_ids = set()  # Track processed post IDs
monitored_posts = {}  # Track all posts to monitor for new comments

def fetch_reddit_data_real_time(my_subreddits, keywords, time_threshold):
    data = []
    current_time = time.time()

    for subreddit_name in my_subreddits:
        subreddit = reddit.subreddit(subreddit_name)
        
        # Fetch new posts
        for submission in subreddit.new(limit=50):
            if submission.created_utc > time_threshold:
                if submission.id not in processed_post_ids:
                    processed_post_ids.add(submission.id)
                    monitored_posts[submission.id] = submission  # Add to monitored posts
                    post_keywords = [kw for kw in keywords if kw.lower() in (submission.title + " " + submission.selftext).lower()]
                    if post_keywords:
                        post_data = {
                            "type": "post",
                            "content": submission.title + " " + submission.selftext,
                            "author": submission.author.name if submission.author else "N/A",
                            "score": submission.score,
                            "created_utc": submission.created_utc,
                            "human_readable_time": datetime.fromtimestamp(submission.created_utc).strftime('%Y-%m-%d %H:%M:%S'),
                            "url": f"https://www.reddit.com{submission.permalink}",
                            "keywords": ', '.join(post_keywords)
                        }
                        data.append(post_data)
    
    # Fetch new comments from monitored posts
    for post_id, submission in monitored_posts.items():
        submission.comments.replace_more(limit=None)
        for comment in submission.comments.list():
            if comment.created_utc > time_threshold:
                comment_keywords = [kw for kw in keywords if kw.lower() in comment.body.lower()]
                if comment_keywords:
                    comment_data = {
                        "type": "comment",
                        "content": comment.body,
                        "author": comment.author.name if comment.author else "N/A",
                        "score": comment.score,
                        "created_utc": comment.created_utc,
                        "human_readable_time": datetime.fromtimestamp(comment.created_utc).strftime('%Y-%m-%d %H:%M:%S'),
                        "url": f"https://www.reddit.com{comment.permalink}",
                        "keywords": ', '.join(comment_keywords)
                    }
                    data.append(comment_data)

    return data, current_time

def simulate_reddit_data(my_subreddits, keywords, batch_interval=120):
    time_threshold = time.time() - (batch_interval * 20)  # Start with older data
    while True:
        data, new_time_threshold = fetch_reddit_data_real_time(my_subreddits, keywords, time_threshold)
        if data:
            timestamp = int(time.time())
            csv_file = os.path.join(output_dir, f'reddit_data_{time.strftime("%d-%m-%Y %H-%M-%S", time.localtime(timestamp))}.csv')
            with open(csv_file, mode='w', newline='', encoding='utf-8') as file:
                writer = csv.DictWriter(file, fieldnames=data[0].keys())
                writer.writeheader()
                writer.writerows(data)
            print(f"Data saved to {csv_file}")
        time_threshold = new_time_threshold
        time.sleep(batch_interval)


# Start the data simulation in a separate thread
threading.Thread(target=simulate_reddit_data, args=(my_subreddits, keywords), daemon=True).start()

# Read the streaming CSV data
reddit_stream = spark \
    .readStream \
    .schema("type STRING, content STRING, author STRING, score INT, created_utc DOUBLE, human_readable_time STRING, url STRING, keywords STRING") \
    .option("header", True) \
    .csv(output_dir)

# Process the data (e.g., filter, select)
processed_data = reddit_stream.select(
    col("type"),
    col("content"),
    col("author"),
    col("score"),
    col("created_utc"),
    col("human_readable_time"),
    col("url"),
    col("keywords")
)

# Write the processed data to CSV
query = processed_data.writeStream \
    .outputMode("append") \
    .format("csv") \
    .option("path", "reddit_stream_processed") \
    .option("checkpointLocation", "checkpoint_dir") \
    .trigger(processingTime='60 seconds') \
    .start()

# Wait for the query to finish
query.awaitTermination()